# 05 - Assistente medico com LangChain (RAG + prontuario)

Demonstra o pipeline do assistente (`src/assistant/chain.py`): consulta ao "prontuario" (SQLite sobre o dataset de stroke), busca semantica (RAG/FAISS) nos protocolos/FAQs, geracao da resposta e aplicacao do guardrail de seguranca. Usa o backend LLM local (fine-tuned) se o adapter da Fase 04 estiver disponivel, caso contrario cai para o Gemini (`src/llm/client.py`), da Fase 2.

In [1]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/stroke-prediction')
    sys.path.insert(0, str(ROOT))
    !pip install -r "{ROOT / 'requirements.txt'}"
    print()
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    sys.path.insert(0, str(ROOT))
    print("Rodando Localmente")

from src.assistant.chain import answer_question
from src.assistant.llm_backend import get_generate_fn, local_adapter_available
from src.assistant.patient_db import build_patient_db
from src.assistant.retriever import build_vectorstore

print('Adapter fine-tuned local disponivel?', local_adapter_available())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.5/257.5 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
build_patient_db()
vectorstore = build_vectorstore()
generate_fn = get_generate_fn()
print('Assistente pronto.')

/content/drive/MyDrive/stroke-prediction/src/assistant/retriever.py:68: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Assistente pronto.


## Pergunta 1 — paciente de alto risco, sobre criterios de trombolise

In [3]:
result = answer_question(patient_id=9046, question='Quais os criterios para considerar trombolise neste paciente?', vectorstore=vectorstore, generate_fn=generate_fn)
print('--- Resposta ---')
print(result['response'])
print()
print('Fontes citadas:', result['sources'])
print('Exames pendentes:', result['pending_exams'])
print('Requer validacao humana?', result['requires_human_validation'])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Resposta ---
Resposta:

Para determinar se o paciente pode ser considerado para a trombólise com alteplase (rt-PA), analiso-se os seguintes critérios internos fictícios do Hospital Fictício:

1. **Diagnóstico Clínico**: O paciente apresenta sinais de déficit neurológico focal agudo, conforme medida pelo NIHSS. No caso específico deste paciente, o valor estimado de peso é de 75 kg (considerando uma média de 70 kg para pacientes com idade entre 60 e 70 anos).

2. **Janela de Tempo**: O início dos sintomas foi reportado como ocorrido há aproximadamente 3 horas, que superam a janela terapêutica padrão de 4,5 horas. Entretanto, este critério é mais rigoroso do que o protocolo real, pois o paciente tem um peso elevado e está sob cuidados intensivos, o que aumenta significativamente a probabilidade de sucesso da trombólise.

3. **Tomografia de Crânio**: Embora não tenha sido mencionado, a tomografia de crânio deveria ser realizada para confirmar a ausência de sangramento ativo cerebral. I

## Pergunta 2 — orientacao a familiares (escala FAST)

In [4]:
result2 = answer_question(patient_id=51676, question='Como explico a escala FAST para a familia deste paciente?', vectorstore=vectorstore, generate_fn=generate_fn)
print(result2['response'])
print()
print('Fontes citadas:', result2['sources'])

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Resposta:

Para explicar a escala FAST para a família do paciente, você pode usar a seguinte estrutura baseada nas instruções fornecidas:

"Segundo o protocolo de FAST desenvolvido pela Associação Americana de Neurologistas, a escala FAST é usada para identificar rapidamente o AVC cerebral. Ela consiste em três pontos: F (Frente), A (Armação), e S (Somato). 

1. **F (Frente)**: Para verificar se o paciente tem dificuldade de expressar sorriso. Peça para ele sorrir e observe se o rosto cai ou se um lado do rosto fica mais baixo. Isso indica que o cérebro esquerdo está afetado.

2. **A (Armação)**: Para ver se o paciente tem dificuldade de levantar os braços. Peça para ele levantar os braços e note se algum deles cairá. Isso indica que o cérebro direito está afetado.

3. **S (Somato)**: Para observar se o paciente tem dificuldade de falar. Peça para ele cantar uma música ou repetir uma frase simples. Note se a fala dele é confusa ou arrastada.

Se qualquer um desses sinais for detectado,

## Explainability

Cada resposta lista as fontes (`sources`) recuperadas pelo RAG que embasaram o texto gerado — atende ao requisito de explainability do desafio (indicar a origem da informacao usada).